In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# XAI Analysis — Grad-CAM on BrugadaCNN\n",
    "**IDSC 2026 | Team MediScript**\n",
    "\n",
    "This notebook documents the Grad-CAM interpretability analysis.\n",
    "\n",
    "## What is Grad-CAM?\n",
    "Gradient-weighted Class Activation Mapping (Grad-CAM) shows which parts of the input\n",
    "the model found most important for its prediction. For ECG signals, this means highlighting\n",
    "which time regions drove the Brugada classification.\n",
    "\n",
    "## Mathematical Formulation\n",
    "Importance weights and saliency map:\n",
    "\n",
    "**αₖ = (1/Z) Σᵢ ∂yᶜ/∂Aᵢₖ**  \n",
    "**Lᶜ = ReLU(Σₖ αₖ · Aₖ)**\n",
    "\n",
    "Maps are upsampled to 1200 samples via linear interpolation and normalized to [0,1].\n",
    "\n",
    "## Target Layer\n",
    "Grad-CAM is applied to the last Conv1d in the encoder — the deepest learned features\n",
    "before Global Average Pooling.\n",
    "\n",
    "## Key Findings\n",
    "| Case | Type | Prob | Grad-CAM Pattern | Clinical Interpretation |\n",
    "|------|------|------|------------------|-------------------------|\n",
    "| Sample 6 | True Positive | 0.999 | Uniform global activation | Persistent spontaneous Brugada pattern |\n",
    "| Sample 30 | True Positive | 0.687 | Beat-aligned periodic peaks | Intermittent/concealed Brugada |\n",
    "| Sample 3 | False Negative | 0.347 | Fragmented, weak activation | Concealed Brugada, not visible at rest |\n",
    "\n",
    "**Key insight:** In all 14 analyzed samples, the model attended to V1–V3 without\n",
    "being told which leads matter — matching clinical diagnostic knowledge.\n",
    "\n",
    "## Generate Grad-CAM Figures\n",
    "To regenerate all Grad-CAM figures, run from the repo root:\n",
    "```bash\n",
    "python scripts/run_gradcam.py\n",
    "```\n",
    "Figures are saved to `outputs/figures/gradcam/`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.image as mpimg\n",
    "import os\n",
    "\n",
    "gradcam_dir = '../outputs/figures/gradcam'\n",
    "samples = [\n",
    "    ('gradcam_sample6_True_Positive.png',   'Sample 6 — True Positive (prob=0.999)'),\n",
    "    ('gradcam_sample30_True_Positive.png',  'Sample 30 — True Positive (prob=0.687)'),\n",
    "    ('gradcam_sample3_False_Negative.png',  'Sample 3 — False Negative (prob=0.347)'),\n",
    "]\n",
    "\n",
    "fig, axes = plt.subplots(3, 1, figsize=(14, 18))\n",
    "for ax, (fname, title) in zip(axes, samples):\n",
    "    path = os.path.join(gradcam_dir, fname)\n",
    "    if os.path.exists(path):\n",
    "        img = mpimg.imread(path)\n",
    "        ax.imshow(img)\n",
    "        ax.set_title(title, fontweight='bold', fontsize=11)\n",
    "        ax.axis('off')\n",
    "    else:\n",
    "        ax.text(0.5, 0.5, f'Run scripts/run_gradcam.py to generate\\n{fname}',\n",
    "                ha='center', va='center', transform=ax.transAxes)\n",
    "        ax.axis('off')\n",
    "\n",
    "plt.suptitle('Grad-CAM Analysis — BrugadaCNN Ensemble', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": { "display_name": "Python 3", "language": "python", "name": "python3" },
  "language_info": { "name": "python", "version": "3.12.0" }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}